# Generate yaml files for the ResBaz website

This script connects to `main-sheet` to pull in data in the sessions, speakers, and schedule sheets to generate yaml files for the ResBaz website.

## 0 - Import libraries and authenticate with a google account

In [ ]:
# import/install packages
import pandas as pd
# import yaml
import yaml
import json
import requests
from pprint import pprint
from google.colab import auth
import gspread
from google.auth import default
from pprint import pprint
from tqdm.auto import tqdm
from tqdm.contrib.concurrent import thread_map
import sys
import re
pd.set_option("display.max_columns", None)
pd.set_option('display.max_colwidth', None)
from google.colab import userdata

In [ ]:
# auth google
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

## 2 - Access `main_sheet`

In [ ]:
def sheet_to_df(sheet):
  rows = sheet.get_all_values()
  df = pd.DataFrame(rows[1:], columns=rows[0])
  return df, df.copy()


# access sessions and schedule sheets and return as dataframe
main_sheet = gc.open_by_key(userdata.get('main_sheet_key'))
sessions_sheet = main_sheet.worksheet('sessions')
schedule_sheet = main_sheet.worksheet('schedule')
speakers_sheet = main_sheet.worksheet("speakers")

schedule_df, schedule_df_copy = sheet_to_df(schedule_sheet)
sessions_df, sessions_df_copy = sheet_to_df(sessions_sheet)
speakers_df, speakers_df_copy = sheet_to_df(speakers_sheet)


schedule_df

## 3 - Generate schedule yaml
- This step relies on confirmed sessions having been added to the `schedule` sheet in the appropriate timeslot.

In [ ]:
schedule_dict = {} # use a dict to store yml structure

grouped = schedule_df.groupby('date') # group df by date
day_list = [] # generate list of dicts for each day
# define dict for each day in schedule
for date, data in grouped:
  day_dict = {}
  day_dict['date'] = date
  day_dict['dateReadable'] = data.dateReadable.unique()[0]
  day_dict['tracks'] = [{"title": "1", "color": "#f27f27"},
  ]
  #only add additional tracks if they have sessions
  if data.track2.str.isnumeric().any():
    day_dict['tracks'].append({"title": "2", "color": "#e91e63"})
  if data.track3.str.isnumeric().any():
    day_dict['tracks'].append({"title": "3", "color": "#3279a8"})
  day_dict['timeslots'] = []
  for i, row in data.iterrows():
    # define timeslots where session exist
    if (row.track1) != '' or (row.track2) != '' or (row.track3) != '':
      day_dict['timeslots'].append({
          'startTime': str(row.startTime),
          'endTime': str(row.endTime),
          'sessionIds': [int(id) for id in [row.track1, row.track2, row.track3] if id]
      })
  day_list.append(day_dict)

day_list

In [ ]:
# Custom class & function to dump data structure in correct format for ResBaz website
class CustomDumper(yaml.SafeDumper):
    def increase_indent(self, flow=False, indentless=False):
      return super(CustomDumper, self).increase_indent(flow, False)
    def represent_data(self, data):
        if isinstance(data, str) and data == '09:00': # handle octals so read as string
            return self.represent_scalar('tag:yaml.org,2002:str', data, style="'")
        return super(CustomDumper, self).represent_data(data)

def dict_representer(dumper, data):
    return dumper.represent_dict(data.items())
def session_representer(dumper, data): # ensure session ids are read correctly.
    return dumper.represent_scalar("tag:yaml.org,2002:int", f"{data:03}")

CustomDumper.add_representer(dict, dict_representer)
CustomDumper.add_representer(int, session_representer)

def custom_dump(data):
    return yaml.dump(data, Dumper=CustomDumper, default_flow_style=None)

# Write the YAML data to a file
with open('schedule.yml', 'w') as file:
    file.write(custom_dump(day_list))

print("YAML file 'schedule.yml' created successfully.")

## 4 - Generate sessions yml
- `instructors` column in `sessions` should contain full names, separated by `,` or `;`, and these need to match names in `speakers`.

In [ ]:
df = pd.DataFrame() # define df of all session ids
df['ids'] =  pd.concat([schedule_df.track3,  schedule_df.track2,  schedule_df.track1])

df = df[df != ''].drop_duplicates().dropna() # drop duplicates and nans
df_sessions_running = sessions_df[sessions_df.id.isin(df.ids)] # If a session ID is in the schedule it's happening

df_sessions_running

In [ ]:
# function to get speaker ids from `sessions`
def get_speakers_id(speaker_names): # function to return speaker ID
  speakers_sheet = main_sheet.worksheet('speakers') # generate lookup table of speaker ids
  rows = speakers_sheet.get_all_values() # provides list of rows
  speakers_df = pd.DataFrame(rows[1:], columns=rows[0])
  speakers_df["fullname"] = speakers_df["name"].str.strip() + " " + speakers_df["surname"].str.strip()
  speakers = []

  for name in speaker_names:
    try:
      speaker = speakers_df[speakers_df["fullname"] == name]
      if speaker.skip.values[0] != "yes":
        speakers.append(int(speaker.id.values[0]))
    except:
       return None
  return speakers

In [ ]:
# generate list of sessions
sessions_dict = [{"id": 200, "title": "Break", "description":'',"speakers": [],"hidden": True},
                 {"id": 201,"title": "Lunch","description":"","speakers": [],"hidden": True},
                 {"id": 300, "title": "To Be Announced", "description":'', "speakers": [], "hidden": True}
                 ] # add fixed sessions required in yml file

for row in df_sessions_running.itertuples(): # iterate over confirmed sessions dataframe.
  if(row.status != "confirmed") and (row.status != "EB-exclude"):#skip excluded events
    continue
  event_dict = {}
  event_dict["id"] = int(row.id)
  event_dict["title"] = row.title
  event_dict["description"] = row.description
  event_dict["subtype"] = "workshop"
  speakers = [i.strip() for i in re.split("[,;]", row.instructors)]
  event_dict["speakers"] = get_speakers_id(speakers)
  event_dict["complexity"] = row.complexity
  event_dict["length"] = f"{row.length} hours"
  event_dict["capacity"] = row.capacity
  event_dict["themes"] = [i.strip().replace("'", '') for i in row.themes.split(",")]
  event_dict["registration_link"] = row.registration_link
  sessions_dict.append(event_dict)

print(sessions_dict)

# Write the YAML data to a file
with open('sessions.yml', 'w') as file:
    file.write(custom_dump(sessions_dict))

print("YAML file 'sessions.yml' created successfully.")

## 5 - Generate speakers yml
- Values in `thumbnailUrl` column in `speakers` must match file paths of images stored in `img/people`.


In [ ]:
# iterate over speaker_df to generate list of dicts
speakers_list = []
for row in speakers_df.itertuples():
  if(row.skip != "yes"):
    speaker_dict = {}
    speaker_dict["id"] = int(row.id)
    speaker_dict["name"] = row.name.strip()
    speaker_dict["surname"] = row.surname.strip()
    speaker_dict["company"] = row.company
    speaker_dict["title"] = row.title
    speaker_dict["bio"] = row.bio
    speaker_dict["thumbnailUrl"] = row.thumbnailUrl
    speaker_dict["rockstar"] = False
    speaker_dict["ribbon"] = [{"abbr": row.ribbon_abbr, "title": row.ribbon_title, "url": row.ribbon_url}]
    speaker_dict["social"] = [{"name": row.social_name, "link": row.social_link}]
    speakers_list.append(speaker_dict)

# Write the YAML data to a file
with open('speakers.yml', 'w', encoding='utf-8') as file:
    file.write(custom_dump(speakers_list))

print("YAML file 'speakers.yml' created successfully.")